# Classification Relationships and Hierarchy — Verification Notebook

**Companion to:** `docs/research/classification-relationships-and-hierarchy.md`

The research document states hypotheses and the evidence available without
row-level data. This notebook runs the tests listed in §8 of that document
against the actual rows, and records a verdict for each one.

**How to use it:** run top to bottom. Every section prints a VERDICT line.
When a section is settled, move the corresponding item in the research document
from §5 *Assumed* to §5 *Demonstrated*, and update the rule status in §6.

**Order matters.** Section 1 cleans the data. Every claim about which label
combinations are impossible is invalid if it runs on uncleaned rows (§2.1).

Priority order, from §8 of the research document:

| Order | Question | Doc ref | Why first |
|---|---|---|---|
| 1 | Is Platform scoped to one manufacturer? | §4.4 | Changes the scope of a target |
| 2 | Is Brand to Mnfr deterministic? | §4.3 | Turns R2 into a hard rule |
| 3 | Is Segment to Sub-Segment strict? | §4.1 | Turns R1 into a hard rule |
| 4 | Full parent-child map | §4.1 | Completes the taxonomy |
| 5 | Platform to Brand map | §4.4 | Supports R3 |
| 6 | Breadcrumb to Segment strength | §4.5 | Baseline + soft rule R5 |
| 7 | What does JoiningKey identify? | §2.6 | Decides the split strategy |

**Where results go.** Derived maps are written to `interim/`, per the repo's data
convention. When a verdict settles a rule (R1 to R6), record it in
`docs/decisions.md` as well - that is where the team keeps agreed rules, and a
rule that only lives in a notebook output is not an agreement.

## 0. Setup

Follows the repo's data convention: the PO's CSVs live in `provided/` under the
project root, with their filenames unchanged, and `provided/`, `interim/` and
`processed/` are all git-ignored.

```
<project root>/
    provided/      Selfcare_Training_data (1).csv
                   Selfcare_Target_data (1).csv
    interim/       <- this notebook writes its maps here
    processed/
```

The cell finds the project root by walking up from the notebook looking for a
`provided/` folder (or a `.git`), so the notebook runs from anywhere in the repo.
If the expected filenames are not there it falls back to detecting the files by
their columns, and if that also fails it prints what it found and stops.

Everything loads as string on purpose: in the target dataset the classification
columns are entirely empty and pandas infers `float64`, which makes a join
against training labels fail silently (§2.4).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

# Leave as None to use the repo convention, or set them directly.
TRAIN_PATH  = None
TARGET_PATH = None

# Filenames as the repo README requires them - do not rename the source files.
TRAIN_NAME  = "Selfcare_Training_data (1).csv"
TARGET_NAME = "Selfcare_Target_data (1).csv"

TARGETS = ["Mnfr", "Brand", "Platform", "Segment", "Sub-Segment", "TargetAgeGroup"]


def find_project_root(start="."):
    """Walk up until we see provided/ (or .git) - that folder is the root."""
    p = Path(start).resolve()
    for _ in range(8):
        if (p / "provided").is_dir() or (p / ".git").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return None


ROOT     = find_project_root()
PROVIDED = (ROOT / "provided") if ROOT else None
INTERIM  = (ROOT / "interim")  if ROOT else Path(".")

print("project root:", ROOT or "not found")
print("provided/   :", PROVIDED if PROVIDED and PROVIDED.is_dir() else "not found")
print()

# 1. the documented filenames, if they are there
if TRAIN_PATH is None and PROVIDED and (PROVIDED / TRAIN_NAME).is_file():
    TRAIN_PATH = str(PROVIDED / TRAIN_NAME)
if TARGET_PATH is None and PROVIDED and (PROVIDED / TARGET_NAME).is_file():
    TARGET_PATH = str(PROVIDED / TARGET_NAME)


SKIP_DIRS = {".git", ".ipynb_checkpoints", "node_modules", "__pycache__",
             ".venv", "venv", "env", "site-packages", "Library", "Applications"}


def find_candidates(start=".", levels=2):
    """CSV files in this folder or up to `levels` parents, nearest first."""
    here = Path(start).resolve()
    roots, p = [], here
    for _ in range(levels + 1):
        roots.append(p)
        if p.parent == p or p == Path.home():
            break
        p = p.parent

    seen = []
    for root in roots:
        for f in root.rglob("*.csv"):
            if SKIP_DIRS & set(f.parts) or f in seen:
                continue
            seen.append(f)
            if len(seen) > 300:      # don't crawl a huge tree
                return sorted(seen, key=lambda q: len(q.relative_to(roots[-1]).parts))
    # nearest to the notebook first
    def dist(q):
        try:
            return len(q.resolve().relative_to(here).parts)
        except ValueError:
            return 10 + len(q.parts)
    return sorted(seen, key=dist)


def header_of(path):
    try:
        return set(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return set()


# 2. fall back to detecting by columns
if TRAIN_PATH is None or TARGET_PATH is None:
    print("Expected filenames not found - falling back to column detection.")
    # Confined to the project root when we found one: never analyse a CSV
    # from outside the repo.
    cands = find_candidates(ROOT, levels=0) if ROOT else find_candidates(".", levels=2)
    matches = []
    for f in cands:
        cols = header_of(f)
        present = [c for c in TARGETS if c in cols]
        if len(present) < 4:          # tolerate a renamed column or two
            continue
        try:
            head = pd.read_csv(f, dtype=str, usecols=present, nrows=5000)
            n = sum(1 for _ in open(f, encoding="utf-8", errors="ignore")) - 1
        except Exception:
            continue
        # The target set is the one whose label columns are EMPTY (§2.4).
        matches.append({"path": f, "rows": n, "cols": len(present),
                        "labelled": bool(head.notna().any(axis=None))})

    # `matches` is already in nearest-first order from find_candidates
    trains  = [m for m in matches if m["labelled"]]
    targets = [m for m in matches if not m["labelled"]]

    if trains and targets:
        TRAIN_PATH  = TRAIN_PATH  or str(trains[0]["path"])
        TARGET_PATH = TARGET_PATH or str(targets[0]["path"])
        print("auto-detected:")
        for m in matches[:8]:
            kind = "labelled (train)" if m["labelled"] else "empty labels (target)"
            print(f"   {m['rows']:>7} rows  {m['cols']}/6 cols  {kind:22s} {m['path']}")
        print()
    else:
        if matches:
            print("Found files with the target columns, but could not tell train from target:")
            for m in matches[:10]:
                print(f"   {m['rows']:>7} rows  labels populated: {m['labelled']}  {m['path']}")
            print()
        print("Could not find the data files.")
        print(f"Expected: {PROVIDED / TRAIN_NAME if PROVIDED else 'provided/' + TRAIN_NAME}")
        print(f"          {PROVIDED / TARGET_NAME if PROVIDED else 'provided/' + TARGET_NAME}")
        print()
        print("The PO's ZIPs are not in GitHub - provided/ is git-ignored.")
        print("Extract them into provided/ with the filenames unchanged.")
        print()
        print(f"Found {len(cands)} CSV file(s):")
        for f in cands[:40]:
            print("   ", f)
        print()
        raise SystemExit(
            "Place the CSVs in provided/, or set TRAIN_PATH and TARGET_PATH above."
        )

print("TRAIN_PATH  =", TRAIN_PATH)
print("TARGET_PATH =", TARGET_PATH)
print()

train  = pd.read_csv(TRAIN_PATH,  dtype=str, keep_default_na=True)
target = pd.read_csv(TARGET_PATH, dtype=str, keep_default_na=True)

print("train ", train.shape, " expected (84552, 32)")
print("target", target.shape, " expected (28082, ...)")
print()
print("targets present:", [c for c in TARGETS if c in train.columns])
print("missing:", [c for c in TARGETS if c not in train.columns])

if len(train) != 84552 or len(target) != 28082:
    print()
    print("!" * 70)
    print("WARNING: row counts do not match the dataset profile.")
    print("Either auto-detection picked the wrong files, or the data has changed")
    print("since the profile was written. Check the paths above before trusting")
    print("anything below - the research document's numbers come from that profile.")
    print("!" * 70)

### 0.1 Sanity check against the dataset profile

If these counts do not match the profile the research document was built from,
stop and find out why before trusting anything below.

In [ ]:
expected = {
    "Mnfr": 78851, "Brand": 84227, "Platform": 1105,
    "Segment": 79760, "Sub-Segment": 77034, "TargetAgeGroup": 77472,
}

rows = []
for c in TARGETS:
    if c not in train.columns:
        continue
    n = train[c].notna().sum()
    rows.append({"target": c, "labeled_rows": n,
                 "profile_expected": expected.get(c),
                 "match": n == expected.get(c)})

pd.DataFrame(rows)

## 1. Cleaning — the malformed rows (§2.1)

The profile reports exactly 3 malformed values in each target, and every one of
them begins with a space. The working hypothesis is that these are the *same*
handful of shifted rows appearing across all six columns.

If that holds, a whitespace filter removes a few rows and costs nothing. If it
does not hold, the corruption is wider than assumed and that is a finding.

In [ ]:
def leading_ws_mask(s):
    return s.notna() & s.str.match(r"^\s")

ws_index = {}
for c in TARGETS:
    if c not in train.columns:
        continue
    m = leading_ws_mask(train[c])
    ws_index[c] = set(train.index[m])
    print(f"{c:16s} {m.sum():3d} malformed  ->  {sorted(train.loc[m, c].unique())}")

all_bad = set().union(*ws_index.values()) if ws_index else set()
print()
print("distinct rows affected across all targets:", len(all_bad))

In [ ]:
# Are they the same rows in every target?  (§2.1 open question 2)
cols = [c for c in TARGETS if c in ws_index]
overlap = pd.DataFrame(
    [[len(ws_index[a] & ws_index[b]) for b in cols] for a in cols],
    index=cols, columns=cols
)
print(overlap)
print()

same_rows = all(ws_index[c] == ws_index[cols[0]] for c in cols)
if same_rows:
    print(f"VERDICT: SAME ROWS. {len(all_bad)} shifted rows explain every malformed "
          f"value in all six targets. Drop them; cost is negligible.")
else:
    print(f"VERDICT: NOT THE SAME ROWS. {len(all_bad)} distinct rows affected. "
          f"The corruption is wider than the research document assumed — "
          f"update §2.1 and investigate the parser.")

In [ ]:
# Build the cleaned frame everything below uses.
clean = train.drop(index=sorted(all_bad)).copy()
for c in TARGETS:
    if c in clean.columns:
        clean[c] = clean[c].str.strip()

print(f"clean: {clean.shape[0]} rows  (dropped {len(all_bad)})")

## 2. Near-duplicate labels (§2.3 / §4.2)

A 4-row class looks like a rare category worth protecting. A spacing variant of
a 1,302-row class is noise that inflates the taxonomy and creates a class no
model can learn. This has to be resolved before the parent-child map is written
down, and it changes the macro-F1 denominator in the evaluation document too.

In [ ]:
def normalize(s):
    return (s.str.lower()
             .str.replace(r"\s*/\s*", "/", regex=True)
             .str.replace(r"\s+", " ", regex=True)
             .str.strip())

for c in ["Sub-Segment", "Segment", "Brand"]:
    if c not in clean.columns:
        continue
    vc = clean[c].value_counts()
    norm = pd.DataFrame({"label": vc.index, "n": vc.values})
    norm["key"] = normalize(norm["label"])
    dupes = norm[norm.duplicated("key", keep=False)].sort_values(["key", "n"],
                                                                ascending=[True, False])
    print(f"--- {c} ---")
    print(dupes.to_string(index=False) if len(dupes) else "no near-duplicates found")
    print()

Candidate pairs named in the research document, for a direct look:

- `Cold/Flu` (1,302) vs `Cold / Flu` (4)
- `Creams/Gels & Medicated Patches` (1,009) vs `Creams & Gels (Rubs)` (1)
- `Other Lifestyle CHC` (202) vs `Other Lifestyle` (3)

Whether these are the same class is a judgement call that needs an owner, not
something the data settles alone. What the data can show is whether the small
variant behaves like the large one — same Segment, similar products.

In [ ]:
pairs = [("Cold/Flu", "Cold / Flu"),
         ("Creams/Gels & Medicated Patches", "Creams & Gels (Rubs)"),
         ("Other Lifestyle CHC", "Other Lifestyle")]

for big, small in pairs:
    sub = clean[clean["Sub-Segment"].isin([big, small])]
    if sub.empty:
        print(f"{big!r} / {small!r}: neither label present after cleaning\n")
        continue
    print(f"--- {big!r}  vs  {small!r} ---")
    print(sub.groupby(["Sub-Segment", "Segment"]).size().rename("n").to_frame())
    cols = [c for c in ["ProductName"] if c in sub.columns]
    if cols:
        print(sub[sub["Sub-Segment"] == small][["Sub-Segment"] + cols].head(10).to_string(index=False))
    print()

## 3. Q1 — Is Platform scoped to one manufacturer? (§4.4)

**This is the highest-impact question in the ticket and it runs first.**

Hypothesis: Platform is a product-line variant within a brand, populated only
for one manufacturer's products. Supporting number from the profile: Platform
has 1,105 labeled rows, Mnfr = J&J has 1,030.

If confirmed, Platform is not sparse at random. It is restricted in scope by
design, the 98.69% missing rate is not a data quality problem, and the right
next step is a scope conversation with the PO rather than a request for more
labels. It also changes the evaluation document's §3.3 conclusion.

In [ ]:
for col in ["Platform", "Mnfr"]:
    if col not in clean.columns:
        raise SystemExit(f"{col} not in this dataset - cannot run section 3")

has_platform = clean[clean["Platform"].notna()]
tab = has_platform["Mnfr"].value_counts(dropna=False).rename("rows_with_platform")
print(tab.to_frame())
print()

total = len(has_platform)
jnj = int(tab.get("J&J", 0))
share = jnj / total if total else 0
print(f"rows with Platform: {total}")
print(f"of those, Mnfr = J&J: {jnj}  ({share:.1%})")
print()

if share >= 0.95:
    print("VERDICT: CONFIRMED. Platform is effectively scoped to one manufacturer.")
    print("  -> Platform is restricted by design, not sparse at random.")
    print("  -> Rule R4 (abstain outside supported scope) is justified.")
    print("  -> Take the scope question to the PO; do not request more labels yet.")
    print("  -> Evaluation document §3.3 needs rewriting on this basis.")
elif share >= 0.7:
    print("VERDICT: PARTIAL. Mostly one manufacturer but not exclusively.")
    print("  -> Look at the non-J&J rows below before concluding.")
else:
    print("VERDICT: REJECTED. Platform is not scoped to one manufacturer.")
    print("  -> The §4.4 hypothesis fails. Platform stays a sparse-label problem")
    print("     and evaluation document §3.3 stands as written.")

# the exceptions are the interesting part either way
others = has_platform[has_platform["Mnfr"] != "J&J"]
if len(others):
    print()
    print(f"--- {len(others)} Platform rows not labeled J&J ---")
    print(others[["Mnfr", "Brand", "Platform"]].head(20).to_string(index=False))

## 4. Q2 — Is Brand to Mnfr deterministic? (§4.3)

Mnfr has only two valid values, so this is a lookup: each brand should map to
exactly one manufacturer. An empty result below means the relationship is
deterministic and rule **R2** graduates from warn mode to a hard rule.

A brand returning 2 is either a labeling inconsistency or a real exception.
Both are worth knowing and neither should be silently dropped.

In [ ]:
bm = clean[clean["Brand"].notna() & clean["Mnfr"].notna()]
n_mnfr = bm.groupby("Brand")["Mnfr"].nunique()
violations = n_mnfr[n_mnfr > 1]

print(f"brands with a manufacturer label: {n_mnfr.size}")
print(f"brands mapping to more than one manufacturer: {len(violations)}")
print()

if len(violations) == 0:
    print("VERDICT: DETERMINISTIC. R2 becomes a hard rule.")
    print("  -> Mnfr is derivable from Brand. A Mnfr model that ignores Brand is")
    print("     solving a harder problem than it needs to (§4.3).")
else:
    print("VERDICT: NOT DETERMINISTIC. R2 stays in warn mode until these are explained.")
    print()
    for b in violations.index:
        print(f"--- {b} ---")
        print(bm[bm["Brand"] == b]["Mnfr"].value_counts().to_string())
        print()

In [ ]:
# The actual lookup table, and which brands belong to J&J.
# Note this comes from the data, not from outside product knowledge (§2, §5).
jnj_brands = (bm[bm["Mnfr"] == "J&J"]
              .groupby("Brand").size().sort_values(ascending=False)
              .rename("rows").to_frame())
print(f"{len(jnj_brands)} brands carry Mnfr = J&J, {int(jnj_brands['rows'].sum())} rows total")
jnj_brands

## 5. Q3 — Is Segment to Sub-Segment a strict hierarchy? (§4.1)

Strict means one parent per child. An empty result means rule **R1** graduates
to a hard rule and the taxonomy is a clean tree.

Run this *after* section 2 is resolved. If `Cold / Flu` and `Cold/Flu` are the
same class written two ways, they may show up here as a false violation, or
mask a real one.

In [ ]:
ss = clean[clean["Segment"].notna() & clean["Sub-Segment"].notna()]
n_parents = ss.groupby("Sub-Segment")["Segment"].nunique()
multi = n_parents[n_parents > 1]

print(f"sub-segments with a parent: {n_parents.size}")
print(f"sub-segments with more than one parent: {len(multi)}")
print()

if len(multi) == 0:
    print("VERDICT: STRICT HIERARCHY. R1 becomes a hard rule.")
else:
    print("VERDICT: NOT STRICT. R1 stays in warn mode.")
    print("  -> Check each case below: some will be the near-duplicate labels")
    print("     from section 2 rather than genuine multi-parent classes.")
    print()
    for s in multi.index:
        print(f"--- {s} ---")
        print(ss[ss["Sub-Segment"] == s]["Segment"].value_counts().to_string())
        print()

### 5.1 The CCFS arithmetic claim

The research document's strongest piece of evidence is that seven Sub-Segments
sum exactly to the CCFS Segment count (4,267). That was derived from the profile
without row-level data. This checks it directly, and also settles whether Nasal
Sprays really sits under CCFS rather than Allergy.

In [ ]:
ccfs_claim = ["Sore Throat / Cough Drops", "Cold/Flu", "Cough", "Nasal Sprays",
              "Sinus excl Sprays", "External Upper Respiratory", "Cold / Flu"]

actual = ss[ss["Segment"] == "CCFS"]["Sub-Segment"].value_counts()
print("Actual children of CCFS:")
print(actual.to_string())
print()
print(f"CCFS rows with a sub-segment: {actual.sum()}")
print()

claimed = set(ccfs_claim)
found = set(actual.index)
print("claimed but not found:", sorted(claimed - found) or "none")
print("found but not claimed:", sorted(found - claimed) or "none")
print()

if claimed == found:
    print("VERDICT: CONFIRMED. The CCFS group is exactly as documented.")
    print("  -> Nasal Sprays belongs under CCFS, not Allergy, as the document argued.")
else:
    print("VERDICT: PARTIALLY WRONG. The arithmetic coincidence misled the profile-only")
    print("  analysis. Correct §4.1 of the research document with the list above.")

### 5.2 The Internal Analgesics coincidence

The research document flagged a 1,065-row gap under Internal Analgesics that
happens to equal the Sleep Aids count exactly, and explicitly said not to write
it down as a finding until tested. This is that test.

In [ ]:
for seg in ["Internal Analgesics", "Allergy"]:
    sub = ss[ss["Segment"] == seg]["Sub-Segment"].value_counts()
    print(f"--- {seg} ---")
    print(sub.to_string())
    print(f"total: {sub.sum()}")
    print()

sleep = ss[ss["Sub-Segment"] == "Sleep Aids"]["Segment"].value_counts()
print("--- Where does Sleep Aids actually sit? ---")
print(sleep.to_string() if len(sleep) else "Sleep Aids not present as a Sub-Segment")

## 6. Q4 — The full parent-child map (§4.1)

This is the artifact the ticket asks for. Export it so the evaluation work and
any future Aryx ontology review can use it directly rather than re-deriving it.

In [ ]:
pc_map = (ss.groupby(["Segment", "Sub-Segment"]).size()
            .rename("n").reset_index()
            .sort_values(["Segment", "n"], ascending=[True, False]))

INTERIM.mkdir(exist_ok=True)
out = INTERIM / "segment_subsegment_map.csv"
pc_map.to_csv(out, index=False)
print(f"{pc_map['Segment'].nunique()} segments, {pc_map['Sub-Segment'].nunique()} sub-segments, "
      f"{len(pc_map)} valid pairs")
print("written to", out)
print()

for seg, g in pc_map.groupby("Segment"):
    print(f"{seg}  ({g['n'].sum()} rows)")
    for _, r in g.iterrows():
        print(f"    {r['Sub-Segment']:45s} {r['n']:6d}")
    print()

### 6.1 Rows with a Sub-Segment but no Segment

The research document noted 4,789 of these, and they are the caveat that keeps
the upper-bound elimination method from being airtight. Worth quantifying, since
they are also recoverable: if a Sub-Segment has exactly one parent, the missing
Segment can be filled in.

In [ ]:
orphans = clean[clean["Sub-Segment"].notna() & clean["Segment"].isna()]
print(f"rows with Sub-Segment but no Segment: {len(orphans)}")
print()
print(orphans["Sub-Segment"].value_counts().head(20).to_string())
print()

single_parent = n_parents[n_parents == 1].index
recoverable = orphans[orphans["Sub-Segment"].isin(single_parent)]
print(f"of those, recoverable from an unambiguous parent: {len(recoverable)} "
      f"({len(recoverable)/len(orphans):.1%})" if len(orphans) else "")

## 7. Q5 — Platform to Brand (§4.4)

Supports rule **R3**. If each Platform value belongs to exactly one Brand, the
rule is hard. The label names suggest it will (Motrin IB under Motrin), but
label-name reading is the weakest method in the document and this is the test
that replaces it.

In [ ]:
pb = clean[clean["Platform"].notna() & clean["Brand"].notna()]
n_brands = pb.groupby("Platform")["Brand"].nunique()
multi_b = n_brands[n_brands > 1]

print(f"platform values with a brand: {n_brands.size}")
print(f"platform values spanning more than one brand: {len(multi_b)}")
print()

if len(multi_b) == 0:
    print("VERDICT: Platform is nested under Brand. R3 becomes a hard rule.")
else:
    print("VERDICT: Platform is not strictly nested under Brand. R3 stays in warn mode.")
    for p in multi_b.index:
        print(f"  {p}: {sorted(pb[pb['Platform'] == p]['Brand'].unique())}")
print()

platform_map = (pb.groupby(["Brand", "Platform"]).size()
                  .rename("n").reset_index()
                  .sort_values(["Brand", "n"], ascending=[True, False]))
INTERIM.mkdir(exist_ok=True)
platform_map.to_csv(INTERIM / "brand_platform_map.csv", index=False)
print("written to", INTERIM / "brand_platform_map.csv")
platform_map

## 8. Q7 — Breadcrumb to Segment strength (§4.5)

`ProductCategory` is a retailer breadcrumb, 99.97% populated. It is not
CatalogIQ's taxonomy, so it cannot be a hard rule. But if a breadcrumb prefix
maps to one Segment more than 90% of the time, that is both a cheap baseline
model and the soft rule **R5**.

It is also the most natural evidence field in the dataset, which the README
requires alongside every classification.

In [ ]:
if "ProductCategory" not in clean.columns:
    raise SystemExit("ProductCategory not in this dataset - skip section 8")

pc = clean[clean["ProductCategory"].notna() & clean["Segment"].notna()].copy()

def prefix(path, depth):
    parts = [p.strip() for p in str(path).split(">")]
    return " > ".join(parts[:depth]) if len(parts) >= depth else None

for depth in (2, 3):
    pc[f"prefix{depth}"] = pc["ProductCategory"].apply(lambda s: prefix(s, depth))

    g = (pc.dropna(subset=[f"prefix{depth}"])
           .groupby([f"prefix{depth}", "Segment"]).size().rename("n").reset_index())
    tot = g.groupby(f"prefix{depth}")["n"].transform("sum")
    g["share"] = g["n"] / tot
    top = (g.sort_values("share", ascending=False)
             .drop_duplicates(f"prefix{depth}")
             .sort_values("n", ascending=False))

    strong = top[(top["share"] >= 0.90) & (top["n"] >= 50)]
    print(f"--- depth {depth}: {len(top)} distinct prefixes, "
          f"{len(strong)} strong (>=90% and >=50 rows) ---")
    print(strong.head(25).to_string(index=False))
    covered = strong["n"].sum()
    print(f"rows covered by strong prefixes: {covered} ({covered/len(pc):.1%})")
    print()

## 9. What does JoiningKey identify? (§2.6)

Not one of the numbered queries, but the evaluation document calls it the
highest-priority open question because it decides the split strategy. Product
catalogs carry near-duplicates — size and flavour variants of the same item —
and a random row split puts near-identical products in both train and test,
inflating every score in a way no metric reveals.

In [ ]:
for key in ["JoiningKey", "Sku", "Upc"]:
    if key not in clean.columns:
        print(f"{key}: not present")
        continue
    s = clean[key]
    n_rows, n_unique = s.notna().sum(), s.nunique()
    print(f"{key:12s} missing {s.isna().mean():6.2%}  "
          f"unique {n_unique:7d}  rows/key {n_rows/max(n_unique,1):.2f}")

print()
if "JoiningKey" in clean.columns:
    grp = clean.groupby("JoiningKey").size().sort_values(ascending=False)
    print("largest JoiningKey groups:")
    print(grp.head(10).to_string())
    print()
    biggest = grp.index[0]
    cols = [c for c in ["ProductName", "Brand", "Segment", "Sub-Segment"] if c in clean.columns]
    print(f"--- rows sharing JoiningKey {biggest!r} ---")
    print(clean[clean["JoiningKey"] == biggest][cols].head(15).to_string(index=False))
    print()
    if grp.max() == 1:
        print("VERDICT: JoiningKey is one row per key. It is a row id, NOT a grouping key.")
        print("  -> It cannot protect against near-duplicate leakage. Look at Sku, or")
        print("     build a grouping key from normalized ProductName + Brand.")
    else:
        print("VERDICT: JoiningKey groups multiple rows. Inspect the rows above:")
        print("  if they are variants of one product, split on this key (§2.6).")

## 10. Summary — what is now demonstrated

Fill this in from the verdicts above, then update the research document:

- Move each settled item from §5 *Assumed* to §5 *Demonstrated*.
- Update the status column in §6 for R1 through R4: hard, or still warn mode.
- If section 3 confirmed the Platform scope hypothesis, rewrite §3.3 of the
  evaluation strategy document, which currently treats Platform as a 61-class
  problem rather than a scope-restricted one.

| Question | Doc ref | Verdict | Rule affected |
|---|---|---|---|
| Malformed rows are the same rows | §2.1 | | — |
| Near-duplicate labels are one class | §4.2 | | — |
| Platform scoped to one manufacturer | §4.4 | | R4 |
| Brand to Mnfr deterministic | §4.3 | | R2 |
| Segment to Sub-Segment strict | §4.1 | | R1 |
| CCFS group as documented | §4.1 | | — |
| Platform nested under Brand | §4.4 | | R3 |
| Breadcrumb prefixes above 90% | §4.5 | | R5 |
| JoiningKey is a grouping key | §2.6 | | — |

**One standard carries over from the research document and does not change here:**
a zero count is *unobserved*, never *impossible*. Only the PO or a taxonomy
document can declare a combination invalid. Rules R5 and R6 must never block.